In [1]:
import pandas as pd
import numpy as np


In [2]:
data = {
    'Customer_ID': ['C001', 'C002', 'C003', 'C004', 'C005'],
    'Age': [25, 34, 45, 23, 52],
    'Gender': ['F', 'M', 'F', 'M', 'F'],
    'Annual_Income': [50000, 60000, 75000, 40000, 85000],
    'Purchase_Amount': [300, 450, 600, 150, 700],
    'Purchase_Frequency': [10, 15, 20, 5, 25]
}

df = pd.DataFrame(data)
print(df)


  Customer_ID  Age Gender  Annual_Income  Purchase_Amount  Purchase_Frequency
0        C001   25      F          50000              300                  10
1        C002   34      M          60000              450                  15
2        C003   45      F          75000              600                  20
3        C004   23      M          40000              150                   5
4        C005   52      F          85000              700                  25


In [3]:
print("\nSummary Statistics:\n", df.describe())

print("\nMissing Values:\n", df.isnull().sum())

print("\nGender Distribution:\n", df['Gender'].value_counts())



Summary Statistics:
              Age  Annual_Income  Purchase_Amount  Purchase_Frequency
count   5.000000       5.000000          5.00000            5.000000
mean   35.800000   62000.000000        440.00000           15.000000
std    12.557866   18234.582529        221.92341            7.905694
min    23.000000   40000.000000        150.00000            5.000000
25%    25.000000   50000.000000        300.00000           10.000000
50%    34.000000   60000.000000        450.00000           15.000000
75%    45.000000   75000.000000        600.00000           20.000000
max    52.000000   85000.000000        700.00000           25.000000

Missing Values:
 Customer_ID           0
Age                   0
Gender                0
Annual_Income         0
Purchase_Amount       0
Purchase_Frequency    0
dtype: int64

Gender Distribution:
 Gender
F    3
M    2
Name: count, dtype: int64


In [4]:
# Add Customer Lifetime Value (CLV)


df['CLV'] = df['Purchase_Amount'] * df['Purchase_Frequency']

# Normalize Income for Purchasing Power Score


income_min = df['Annual_Income'].min()
income_max = df['Annual_Income'].max()
income_norm = (df['Annual_Income'] - income_min) / (income_max - income_min)

df['Power_Score'] = np.round(income_norm * df['Purchase_Frequency'], 2)

print("\nData with Derived Metrics:\n", df)



Data with Derived Metrics:
   Customer_ID  Age Gender  Annual_Income  Purchase_Amount  Purchase_Frequency  \
0        C001   25      F          50000              300                  10   
1        C002   34      M          60000              450                  15   
2        C003   45      F          75000              600                  20   
3        C004   23      M          40000              150                   5   
4        C005   52      F          85000              700                  25   

     CLV  Power_Score  
0   3000         2.22  
1   6750         6.67  
2  12000        15.56  
3    750         0.00  
4  17500        25.00  


In [5]:
avg_purchase_by_gender = df.groupby('Gender')['Purchase_Amount'].mean()
print("\nAverage Purchase Amount by Gender:\n", avg_purchase_by_gender)

high_clv = df[df['CLV'] > df['CLV'].mean()]
print("\nCustomers with High CLV:\n", high_clv)



Average Purchase Amount by Gender:
 Gender
F    533.333333
M    300.000000
Name: Purchase_Amount, dtype: float64

Customers with High CLV:
   Customer_ID  Age Gender  Annual_Income  Purchase_Amount  Purchase_Frequency  \
2        C003   45      F          75000              600                  20   
4        C005   52      F          85000              700                  25   

     CLV  Power_Score  
2  12000        15.56  
4  17500        25.00  


In [6]:
# Top 3 customers by CLV


top_customers = df.sort_values(by='CLV', ascending=False).head(3)
print("\nTop Customers by CLV:\n", top_customers)

# Filter: Age > 30 and Power Score > 10


filtered_customers = df[(df['Age'] > 30) & (df['Power_Score'] > 10)]
print("\nFiltered Customers (Age > 30 and Power Score > 10):\n", filtered_customers)



Top Customers by CLV:
   Customer_ID  Age Gender  Annual_Income  Purchase_Amount  Purchase_Frequency  \
4        C005   52      F          85000              700                  25   
2        C003   45      F          75000              600                  20   
1        C002   34      M          60000              450                  15   

     CLV  Power_Score  
4  17500        25.00  
2  12000        15.56  
1   6750         6.67  

Filtered Customers (Age > 30 and Power Score > 10):
   Customer_ID  Age Gender  Annual_Income  Purchase_Amount  Purchase_Frequency  \
2        C003   45      F          75000              600                  20   
4        C005   52      F          85000              700                  25   

     CLV  Power_Score  
2  12000        15.56  
4  17500        25.00  


In [7]:
df.to_csv('customer_analysis_output.csv', index=False)


In [8]:
# Create CLV Categories


def clv_category(clv):
    if clv >= 5000:
        return 'High'
    elif clv >= 2000:
        return 'Medium'
    else:
        return 'Low'

df['CLV_Category'] = df['CLV'].apply(clv_category)
print("\nCLV Categories:\n", df[['Customer_ID', 'CLV', 'CLV_Category']])



CLV Categories:
   Customer_ID    CLV CLV_Category
0        C001   3000       Medium
1        C002   6750         High
2        C003  12000         High
3        C004    750          Low
4        C005  17500         High


In [9]:
# Segment by Income


income_bins = [0, 50000, 70000, 100000]
income_labels = ['Low Income', 'Middle Income', 'High Income']
df['Income_Segment'] = pd.cut(df['Annual_Income'], bins=income_bins, labels=income_labels)

income_summary = df.groupby('Income_Segment')['Purchase_Amount'].mean()
print("\nAverage Purchase Amount by Income Segment:\n", income_summary)



Average Purchase Amount by Income Segment:
 Income_Segment
Low Income       225.0
Middle Income    450.0
High Income      650.0
Name: Purchase_Amount, dtype: float64


<ipython-input-9-61a59cd4b0f8>:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  income_summary = df.groupby('Income_Segment')['Purchase_Amount'].mean()


In [10]:
# Correlation between numeric columns


correlation_matrix = df[['Age', 'Annual_Income', 'Purchase_Amount', 'Purchase_Frequency', 'CLV', 'Power_Score']].corr()
print("\nCorrelation Matrix:\n", correlation_matrix)



Correlation Matrix:
                          Age  Annual_Income  Purchase_Amount  \
Age                 1.000000       0.990229         0.976897   
Annual_Income       0.990229       1.000000         0.994642   
Purchase_Amount     0.976897       0.994642         1.000000   
Purchase_Frequency  0.982084       0.997176         0.997459   
CLV                 0.993546       0.991870         0.976323   
Power_Score         0.986314       0.977835         0.953012   

                    Purchase_Frequency       CLV  Power_Score  
Age                           0.982084  0.993546     0.986314  
Annual_Income                 0.997176  0.991870     0.977835  
Purchase_Amount               0.997459  0.976323     0.953012  
Purchase_Frequency            1.000000  0.987105     0.968780  
CLV                           0.987105  1.000000     0.995746  
Power_Score                   0.968780  0.995746     1.000000  


In [11]:
# Pivot Table: Average CLV by Gender and Income Segment


pivot = pd.pivot_table(df, values='CLV', index='Gender', columns='Income_Segment', aggfunc='mean')
print("\nPivot Table - Avg CLV by Gender & Income Segment:\n", pivot)



Pivot Table - Avg CLV by Gender & Income Segment:
 Income_Segment  Low Income  Middle Income  High Income
Gender                                                
F                   3000.0            NaN      14750.0
M                    750.0         6750.0          NaN


<ipython-input-11-ae7e45f76d0a>:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = pd.pivot_table(df, values='CLV', index='Gender', columns='Income_Segment', aggfunc='mean')


In [12]:
# Standard Deviation & Median


std_dev_income = np.std(df['Annual_Income'])
median_income = np.median(df['Annual_Income'])
print("\nStandard Deviation of Income:", std_dev_income)
print("Median Income:", median_income)



Standard Deviation of Income: 16309.50643030009
Median Income: 60000.0


In [13]:
# Save high CLV customers separately


df[df['CLV_Category'] == 'High'].to_csv('high_clv_customers.csv', index=False)
